# Tutorial 8: Photometric Calibration

This tutorial demonstrates photometric calibration and systematic correction techniques in brutus, essential for accurate stellar parameter estimation.

## Topics Covered

1. **Understanding photometric offsets** and their sources
2. **Deriving empirical calibrations** from cluster data
3. **Cross-survey calibration** and transformations
4. **Systematic error modeling** and correction
5. **Complete calibration workflow** implementation

## Prerequisites

This tutorial uses:
- MIST isochrones for cluster fitting
- Synthetic data for demonstrations
- Optional: Pre-computed offset files

No special data downloads are required - the tutorial will generate synthetic data as needed.

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import tutorial utilities
from tutorial_utils import (
    set_plot_style,
    find_brutus_data_file,
    save_figure as save_fig_util,
    print_section
)

# Set plot style
set_plot_style()
plt.rcParams['figure.figsize'] = (10, 6)

# Create plots directory if needed
plots_dir = Path('plots/tutorial_08')
plots_dir.mkdir(parents=True, exist_ok=True)

def save_figure(fig, name):
    """Helper to save figures."""
    filepath = plots_dir / f"{name}.png"
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"  Saved: {filepath}")

## Section 1: Understanding Photometric Offsets

Photometric offsets are systematic differences between observed and model magnitudes. They arise from:

### Sources of Offsets
- **Filter profile differences**: Real vs theoretical filter curves
- **Calibration systematics**: Zero-point errors
- **Atmospheric model differences**: Synthetic vs real stellar atmospheres
- **Metallicity effects**: Model grid limitations

### Impact on Science
- Age estimates: ~0.1 Gyr per 1% offset
- Distance: ~10 pc per 1% offset
- [Fe/H]: ~0.05 dex per 1% color offset

Let's visualize typical photometric offsets.

In [ ]:
# Demonstrate typical photometric offsets for different surveys
print("Visualizing typical photometric offsets...\n")

# Synthetic offset data for different surveys
offset_data = {
    'Pan-STARRS': {
        'bands': ['g', 'r', 'i', 'z', 'y'],
        'offsets': [0.015, -0.008, 0.003, 0.012, -0.020],
        'errors': [0.003, 0.002, 0.002, 0.003, 0.005]
    },
    '2MASS': {
        'bands': ['J', 'H', 'Ks'],
        'offsets': [0.025, 0.018, 0.010],
        'errors': [0.005, 0.005, 0.006]
    },
    'Gaia DR3': {
        'bands': ['G', 'BP', 'RP'],
        'offsets': [-0.002, 0.008, -0.005],
        'errors': [0.002, 0.003, 0.003]
    }
}

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (survey, data) in enumerate(offset_data.items()):
    ax = axes[idx]
    
    bands = data['bands']
    offsets = data['offsets']
    errors = data['errors']
    
    # Create bar plot
    x = np.arange(len(bands))
    bars = ax.bar(x, offsets, yerr=errors, capsize=5, alpha=0.7)
    
    # Color bars by offset sign
    for bar, offset in zip(bars, offsets):
        if offset > 0:
            bar.set_color('red')
        else:
            bar.set_color('blue')
    
    ax.set_xticks(x)
    ax.set_xticklabels(bands)
    ax.set_ylabel('Offset (mag)')
    ax.set_title(f'{survey} Photometric Offsets')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0, color='k', linestyle='-', linewidth=1, alpha=0.5)
    ax.set_ylim(-0.04, 0.04)
    
    # Add text annotation for RMS
    rms = np.sqrt(np.mean(np.array(offsets)**2))
    ax.text(0.95, 0.95, f'RMS: {rms:.3f} mag',
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Typical Photometric Offsets by Survey', fontsize=14, fontweight='bold')
save_figure(fig, 'photometric_offsets')
plt.show()

print("\nKey observations:")
print("  • Offsets typically range from -0.02 to +0.02 mag")
print("  • Different signs indicate over/under-estimation")
print("  • NIR bands often have larger offsets than optical")
print("  • Precise calibration is crucial for accurate stellar parameters")

## Section 2: Deriving Empirical Calibrations from Cluster Data

Star clusters provide ideal calibrators because all members share:
- Same age
- Same metallicity
- Same distance
- Minimal differential reddening

We can use isochrone fitting to derive photometric offsets by minimizing residuals between observed and model CMDs.

In [ ]:
# Generate synthetic cluster data for calibration
print("Generating synthetic cluster for calibration...\n")

# True cluster parameters (Hyades-like)
true_params = {
    'age': 625e6,      # 625 Myr
    '[Fe/H]': 0.15,    # Slightly metal-rich
    'distance': 47.0,  # 47 pc
    'A_V': 0.0         # No extinction
}

print(f"Cluster parameters:")
for key, value in true_params.items():
    if key == 'age':
        print(f"  {key}: {value/1e6:.0f} Myr")
    else:
        print(f"  {key}: {value}")

# Generate synthetic isochrone points
n_stars = 100
masses = np.random.uniform(0.5, 1.5, n_stars)  # Solar-like stars

# Simple main sequence relations (approximations)
log_teff = 3.8 - 0.2 * (1.0 - masses)
log_L = 4 * (masses - 1.0)  # Simplified L ∝ M^4

# Convert to magnitudes (simplified)
Mbol = 4.75 - 2.5 * log_L
dm = 5 * np.log10(true_params['distance'] / 10)

# Generate synthetic photometry in multiple bands
bands = ['g', 'r', 'i', 'z', 'y']
true_mags = {}

for i, band in enumerate(bands):
    # Simple color relations (approximations)
    if band == 'g':
        BC = -0.5 * (log_teff - 3.76)  # Bolometric correction
    elif band == 'r':
        BC = -0.3 * (log_teff - 3.76)
    elif band == 'i':
        BC = -0.2 * (log_teff - 3.76)
    elif band == 'z':
        BC = -0.15 * (log_teff - 3.76)
    else:  # y
        BC = -0.1 * (log_teff - 3.76)
    
    true_mags[band] = Mbol - BC + dm

# Add systematic offsets (what we want to recover)
true_offsets = {
    'g': 0.020,
    'r': -0.010,
    'i': 0.000,
    'z': 0.010,
    'y': -0.015
}

# Create observed photometry
obs_mags = {}
obs_errors = {}

for band in bands:
    # Add systematic offset
    obs_mags[band] = true_mags[band] + true_offsets[band]
    # Add random photometric errors
    photometric_error = 0.02  # 0.02 mag errors
    obs_mags[band] += np.random.normal(0, photometric_error, n_stars)
    obs_errors[band] = np.full(n_stars, photometric_error)

print(f"\nGenerated {n_stars} synthetic cluster stars")
print(f"Applied systematic offsets to simulate calibration problem")

In [ ]:
# Derive offsets by isochrone fitting
print("\nDeriving offsets by isochrone fitting...\n")

# Define simple chi-square function
def calculate_chi2(offsets, obs_mags, true_mags, obs_errors):
    """Calculate chi-square between observed (corrected) and model."""
    chi2 = 0
    for i, band in enumerate(bands):
        # Apply offset correction
        corrected = obs_mags[band] - offsets[i]
        # Calculate chi-square
        residuals = corrected - true_mags[band]
        chi2 += np.sum((residuals / obs_errors[band])**2)
    return chi2

# Optimize offsets
initial_offsets = np.zeros(len(bands))  # Start with no offsets

result = minimize(
    calculate_chi2,
    initial_offsets,
    args=(obs_mags, true_mags, obs_errors),
    method='Nelder-Mead',
    options={'maxiter': 1000}
)

derived_offsets = result.x

# Compare derived vs true offsets
print("Offset recovery:")
print("  Band   True    Derived   Difference")
print("  " + "-"*40)
for i, band in enumerate(bands):
    diff = derived_offsets[i] - true_offsets[band]
    print(f"  {band:4s}  {true_offsets[band]:+6.3f}  {derived_offsets[i]:+6.3f}    {diff:+6.3f}")

rms_error = np.sqrt(np.mean([(derived_offsets[i] - true_offsets[band])**2 
                             for i, band in enumerate(bands)]))
print(f"\nRMS recovery error: {rms_error:.4f} mag")

In [ ]:
# Visualize the calibration results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Panel 1: Original CMD (with offsets)
ax = axes[0, 0]
color_orig = obs_mags['g'] - obs_mags['r']
ax.scatter(color_orig, obs_mags['r'], alpha=0.5, s=20, c='blue', label='Observed')
ax.set_xlabel('g - r (mag)')
ax.set_ylabel('r (mag)')
ax.set_title('Original CMD (with offsets)')
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
ax.legend()

# Panel 2: Corrected CMD
ax = axes[0, 1]
g_corr = obs_mags['g'] - derived_offsets[0]
r_corr = obs_mags['r'] - derived_offsets[1]
color_corr = g_corr - r_corr
ax.scatter(color_corr, r_corr, alpha=0.5, s=20, c='green', label='Corrected')
# Overplot true positions
true_color = true_mags['g'] - true_mags['r']
ax.scatter(true_color, true_mags['r'], alpha=0.3, s=10, c='red', label='True')
ax.set_xlabel('g - r (mag)')
ax.set_ylabel('r (mag)')
ax.set_title('Corrected CMD')
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
ax.legend()

# Panel 3: Offset comparison
ax = axes[0, 2]
x = np.arange(len(bands))
width = 0.35
true_vals = [true_offsets[b] for b in bands]
derived_vals = derived_offsets

bars1 = ax.bar(x - width/2, true_vals, width, label='True', alpha=0.7, color='red')
bars2 = ax.bar(x + width/2, derived_vals, width, label='Derived', alpha=0.7, color='green')

ax.set_xlabel('Band')
ax.set_ylabel('Offset (mag)')
ax.set_title('Offset Comparison')
ax.set_xticks(x)
ax.set_xticklabels(bands)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='k', linestyle='-', alpha=0.5)

# Panels 4-6: Color-color diagrams
colors_list = [
    ('g-r', 'r-i', obs_mags['g'] - obs_mags['r'], obs_mags['r'] - obs_mags['i']),
    ('r-i', 'i-z', obs_mags['r'] - obs_mags['i'], obs_mags['i'] - obs_mags['z']),
    ('i-z', 'z-y', obs_mags['i'] - obs_mags['z'], obs_mags['z'] - obs_mags['y'])
]

for idx, (xlab, ylab, xdata, ydata) in enumerate(colors_list):
    ax = axes[1, idx]
    
    # Original
    ax.scatter(xdata, ydata, alpha=0.3, s=20, c='blue', label='Original')
    
    # Corrected colors
    if idx == 0:  # g-r vs r-i
        x_corr = (obs_mags['g'] - derived_offsets[0]) - (obs_mags['r'] - derived_offsets[1])
        y_corr = (obs_mags['r'] - derived_offsets[1]) - (obs_mags['i'] - derived_offsets[2])
    elif idx == 1:  # r-i vs i-z
        x_corr = (obs_mags['r'] - derived_offsets[1]) - (obs_mags['i'] - derived_offsets[2])
        y_corr = (obs_mags['i'] - derived_offsets[2]) - (obs_mags['z'] - derived_offsets[3])
    else:  # i-z vs z-y
        x_corr = (obs_mags['i'] - derived_offsets[2]) - (obs_mags['z'] - derived_offsets[3])
        y_corr = (obs_mags['z'] - derived_offsets[3]) - (obs_mags['y'] - derived_offsets[4])
    
    ax.scatter(x_corr, y_corr, alpha=0.5, s=20, c='green', label='Corrected')
    
    ax.set_xlabel(f'{xlab} (mag)')
    ax.set_ylabel(f'{ylab} (mag)')
    ax.set_title(f'Color-Color: {xlab} vs {ylab}')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

plt.suptitle('Empirical Calibration from Cluster Data', fontsize=14, fontweight='bold')
save_figure(fig, 'cluster_calibration')
plt.show()

print("\n✓ Successfully demonstrated empirical calibration")

## Section 3: Cross-Survey Calibration

Different photometric surveys use different:
- Filter systems
- Calibration procedures
- Photometric systems

Cross-survey calibration involves deriving transformation equations, typically including color terms:

Survey1 - Survey2 = a + b × (color)

Let's demonstrate this with synthetic multi-survey observations.

In [ ]:
# Simulate stars observed by multiple surveys
print("Simulating multi-survey observations...\n")

n_stars = 500

# True stellar properties
true_gmag = np.random.uniform(14, 18, n_stars)
true_color_gr = np.random.uniform(-0.2, 1.5, n_stars)
true_color_ri = 0.5 * true_color_gr + np.random.normal(0, 0.1, n_stars)

# True magnitudes
true_r = true_gmag - true_color_gr
true_i = true_r - true_color_ri

# Survey 1 (e.g., Pan-STARRS)
ps1_offsets = {'g': 0.010, 'r': -0.005, 'i': 0.003}
ps1_color_terms = {'g': 0.02, 'r': -0.01, 'i': 0.005}  # Color-dependent
ps1_errors = 0.015

ps1_g = true_gmag + ps1_offsets['g'] + ps1_color_terms['g'] * true_color_gr + np.random.normal(0, ps1_errors, n_stars)
ps1_r = true_r + ps1_offsets['r'] + ps1_color_terms['r'] * true_color_gr + np.random.normal(0, ps1_errors, n_stars)
ps1_i = true_i + ps1_offsets['i'] + ps1_color_terms['i'] * true_color_ri + np.random.normal(0, ps1_errors, n_stars)

# Survey 2 (e.g., SDSS)
sdss_offsets = {'g': -0.020, 'r': 0.010, 'i': -0.010}
sdss_color_terms = {'g': -0.03, 'r': 0.015, 'i': -0.008}
sdss_errors = 0.020

sdss_g = true_gmag + sdss_offsets['g'] + sdss_color_terms['g'] * true_color_gr + np.random.normal(0, sdss_errors, n_stars)
sdss_r = true_r + sdss_offsets['r'] + sdss_color_terms['r'] * true_color_gr + np.random.normal(0, sdss_errors, n_stars)
sdss_i = true_i + sdss_offsets['i'] + sdss_color_terms['i'] * true_color_ri + np.random.normal(0, sdss_errors, n_stars)

print(f"Generated {n_stars} stars observed by both surveys")
print(f"\nSurvey characteristics:")
print(f"  Pan-STARRS: σ = {ps1_errors:.3f} mag")
print(f"  SDSS:       σ = {sdss_errors:.3f} mag")

In [ ]:
# Derive cross-survey transformations
print("\nDeriving cross-survey transformations...\n")

# Calculate differences
diff_g = ps1_g - sdss_g
diff_r = ps1_r - sdss_r
diff_i = ps1_i - sdss_i

# Use PS1 color as reference
color_ps1 = ps1_g - ps1_r

# Fit linear transformations: PS1 - SDSS = a + b*(g-r)
transformations = {}

for band, diff in [('g', diff_g), ('r', diff_r), ('i', diff_i)]:
    # Remove outliers (3-sigma clipping)
    mask = np.abs(diff - np.median(diff)) < 3 * np.std(diff)
    
    # Linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        color_ps1[mask], diff[mask]
    )
    
    transformations[band] = {
        'slope': slope,
        'intercept': intercept,
        'r_squared': r_value**2,
        'std_err': std_err
    }
    
    print(f"{band}-band: PS1 - SDSS = {intercept:+.4f} + {slope:+.4f} × (g-r)")
    print(f"         R² = {r_value**2:.3f}, σ_fit = {std_err:.4f} mag")
    print()

In [ ]:
# Visualize cross-survey calibration
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Top row: Magnitude differences vs color
bands = ['g', 'r', 'i']
diffs = [diff_g, diff_r, diff_i]

for idx, (band, diff) in enumerate(zip(bands, diffs)):
    ax = axes[0, idx]
    
    # Scatter plot
    sc = ax.scatter(color_ps1, diff, alpha=0.3, s=10, c=ps1_r,
                   cmap='viridis', vmin=14, vmax=18)
    
    # Fit line
    trans = transformations[band]
    x_fit = np.linspace(color_ps1.min(), color_ps1.max(), 100)
    y_fit = trans['intercept'] + trans['slope'] * x_fit
    ax.plot(x_fit, y_fit, 'r-', linewidth=2,
           label=f"y = {trans['intercept']:.3f} + {trans['slope']:.3f}x")
    
    # 1-sigma scatter
    residuals = diff - (trans['intercept'] + trans['slope'] * color_ps1)
    sigma = np.std(residuals)
    ax.fill_between(x_fit, y_fit - sigma, y_fit + sigma, alpha=0.2, color='red')
    
    ax.set_xlabel('PS1 (g - r) (mag)')
    ax.set_ylabel(f'PS1 - SDSS ({band}) (mag)')
    ax.set_title(f'{band}-band Transformation')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    
    if idx == 2:
        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label('PS1 r (mag)')

# Bottom row: Analysis plots

# Panel 1: Direct magnitude comparison
ax = axes[1, 0]
ax.scatter(sdss_r, ps1_r, alpha=0.3, s=10)
ax.plot([14, 18], [14, 18], 'r--', alpha=0.5, label='1:1 line')
ax.set_xlabel('SDSS r (mag)')
ax.set_ylabel('PS1 r (mag)')
ax.set_title('Direct Magnitude Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Color comparison
ax = axes[1, 1]
color_sdss = sdss_g - sdss_r
ax.scatter(color_sdss, color_ps1, alpha=0.3, s=10)
ax.plot([-0.5, 2], [-0.5, 2], 'r--', alpha=0.5, label='1:1 line')
ax.set_xlabel('SDSS (g - r) (mag)')
ax.set_ylabel('PS1 (g - r) (mag)')
ax.set_title('Color Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Residual distributions
ax = axes[1, 2]
for band, diff in zip(bands, diffs):
    trans = transformations[band]
    residuals = diff - (trans['intercept'] + trans['slope'] * color_ps1)
    ax.hist(residuals, bins=30, alpha=0.5, label=f'{band}-band', density=True)

ax.set_xlabel('Residual (mag)')
ax.set_ylabel('Density')
ax.set_title('Transformation Residuals')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axvline(x=0, color='k', linestyle='--', alpha=0.5)

plt.suptitle('Cross-Survey Calibration: Pan-STARRS vs SDSS', fontsize=14, fontweight='bold')
save_figure(fig, 'cross_survey_calibration')
plt.show()

print("✓ Cross-survey calibration complete")

## Section 4: Systematic Error Modeling and Correction

Systematic errors can depend on multiple factors:
- **Magnitude**: Non-linearity in detector response
- **Color**: Atmospheric/filter effects
- **Position**: Flat-fielding, vignetting
- **Time**: Atmospheric variations

Let's model and correct complex systematic patterns.

In [ ]:
# Simulate complex systematic errors
print("Simulating systematic error patterns...\n")

# Create a grid of observations
mag_grid = np.linspace(14, 20, 50)
color_grid = np.linspace(-0.5, 2.0, 50)
MAG, COLOR = np.meshgrid(mag_grid, color_grid)

# Systematic error components:

# 1. Magnitude-dependent (non-linearity)
mag_sys = 0.002 * (MAG - 17)**2

# 2. Color-dependent (atmospheric/filter effects)
color_sys = 0.01 * (COLOR - 0.7)

# 3. Spatial variation (simplified as radial pattern)
radius = np.sqrt((MAG - 17)**2 + (COLOR - 0.7)**2) / 5
spatial_sys = 0.005 * np.sin(2 * np.pi * radius)

# 4. Cross-term
cross_sys = 0.001 * (MAG - 17) * (COLOR - 0.7)

# Total systematic error
total_sys = mag_sys + color_sys + spatial_sys + cross_sys

print("Systematic error components:")
print(f"  Magnitude-dependent: max = {np.abs(mag_sys).max():.3f} mag")
print(f"  Color-dependent:     max = {np.abs(color_sys).max():.3f} mag")
print(f"  Spatial:             max = {np.abs(spatial_sys).max():.3f} mag")
print(f"  Cross-term:          max = {np.abs(cross_sys).max():.3f} mag")
print(f"  Total:               max = {np.abs(total_sys).max():.3f} mag")

In [ ]:
# Visualize systematic error patterns
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Common colormap settings
vmax = 0.04
levels = 20

# Plot 1: Magnitude-dependent systematics
ax = axes[0, 0]
im1 = ax.contourf(MAG, COLOR, mag_sys, levels=levels, cmap='RdBu_r',
                  vmin=-vmax, vmax=vmax)
ax.set_xlabel('Magnitude (mag)')
ax.set_ylabel('Color (mag)')
ax.set_title('Magnitude-Dependent Systematics')
plt.colorbar(im1, ax=ax, label='Error (mag)')

# Plot 2: Color-dependent systematics
ax = axes[0, 1]
im2 = ax.contourf(MAG, COLOR, color_sys, levels=levels, cmap='RdBu_r',
                  vmin=-vmax, vmax=vmax)
ax.set_xlabel('Magnitude (mag)')
ax.set_ylabel('Color (mag)')
ax.set_title('Color-Dependent Systematics')
plt.colorbar(im2, ax=ax, label='Error (mag)')

# Plot 3: Spatial systematics
ax = axes[1, 0]
im3 = ax.contourf(MAG, COLOR, spatial_sys, levels=levels, cmap='RdBu_r',
                  vmin=-vmax/2, vmax=vmax/2)
ax.set_xlabel('Magnitude (mag)')
ax.set_ylabel('Color (mag)')
ax.set_title('Spatial Systematics')
plt.colorbar(im3, ax=ax, label='Error (mag)')

# Plot 4: Total systematics
ax = axes[1, 1]
im4 = ax.contourf(MAG, COLOR, total_sys, levels=levels, cmap='RdBu_r',
                  vmin=-vmax, vmax=vmax)
ax.set_xlabel('Magnitude (mag)')
ax.set_ylabel('Color (mag)')
ax.set_title('Total Systematic Errors')
plt.colorbar(im4, ax=ax, label='Error (mag)')

plt.suptitle('Systematic Error Patterns', fontsize=14, fontweight='bold')
save_figure(fig, 'systematic_patterns')
plt.show()

print("\n✓ Visualized systematic error patterns")

In [ ]:
# Demonstrate systematic error correction
print("\nDemonstrating systematic error correction...\n")

# Generate mock observations with systematics
n_cal = 1000  # Calibration stars
true_mag_cal = np.random.uniform(14, 20, n_cal)
true_color_cal = np.random.uniform(-0.5, 2.0, n_cal)

# Add systematics
mag_sys_cal = 0.002 * (true_mag_cal - 17)**2
color_sys_cal = 0.01 * (true_color_cal - 0.7)
radius_cal = np.sqrt((true_mag_cal - 17)**2 + (true_color_cal - 0.7)**2) / 5
spatial_sys_cal = 0.005 * np.sin(2 * np.pi * radius_cal)
cross_sys_cal = 0.001 * (true_mag_cal - 17) * (true_color_cal - 0.7)
total_sys_cal = mag_sys_cal + color_sys_cal + spatial_sys_cal + cross_sys_cal

# Observed = True + Systematic + Random
obs_mag_cal = true_mag_cal + total_sys_cal + np.random.normal(0, 0.01, n_cal)

# Fit correction model (polynomial)
def sys_model(X, a0, a1, a2, a3, a4, a5, a6):
    """Systematic error model."""
    mag, color = X
    return (a0 + 
            a1 * (mag - 17) + 
            a2 * (mag - 17)**2 + 
            a3 * (color - 0.7) + 
            a4 * (color - 0.7)**2 +
            a5 * (mag - 17) * (color - 0.7) +
            a6 * np.sin(2 * np.pi * np.sqrt((mag - 17)**2 + (color - 0.7)**2) / 5))

# Fit to residuals
residuals = obs_mag_cal - true_mag_cal
popt, pcov = curve_fit(sys_model, (true_mag_cal, true_color_cal), residuals)

print("Fitted correction parameters:")
param_names = ['a0 (const)', 'a1 (mag)', 'a2 (mag²)', 'a3 (color)',
               'a4 (color²)', 'a5 (cross)', 'a6 (spatial)']
for name, value in zip(param_names, popt):
    print(f"  {name:12s}: {value:+.6f}")

# Apply correction
correction = sys_model((true_mag_cal, true_color_cal), *popt)
corrected_mag = obs_mag_cal - correction

# Evaluate improvement
rms_before = np.sqrt(np.mean((obs_mag_cal - true_mag_cal)**2))
rms_after = np.sqrt(np.mean((corrected_mag - true_mag_cal)**2))

print(f"\nCorrection results:")
print(f"  RMS error before correction: {rms_before:.4f} mag")
print(f"  RMS error after correction:  {rms_after:.4f} mag")
print(f"  Improvement factor: {rms_before/rms_after:.2f}×")

In [ ]:
# Visualize the correction results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Before correction
ax = axes[0]
sc = ax.scatter(true_mag_cal, obs_mag_cal - true_mag_cal,
               c=true_color_cal, s=5, alpha=0.5, cmap='viridis',
               vmin=-0.5, vmax=2.0)
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('True Magnitude (mag)')
ax.set_ylabel('Observed - True (mag)')
ax.set_title(f'Before Correction (RMS={rms_before:.4f})')
ax.set_ylim(-0.1, 0.1)
ax.grid(True, alpha=0.3)

# Panel 2: After correction
ax = axes[1]
ax.scatter(true_mag_cal, corrected_mag - true_mag_cal,
          c=true_color_cal, s=5, alpha=0.5, cmap='viridis',
          vmin=-0.5, vmax=2.0)
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('True Magnitude (mag)')
ax.set_ylabel('Corrected - True (mag)')
ax.set_title(f'After Correction (RMS={rms_after:.4f})')
ax.set_ylim(-0.1, 0.1)
ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label='Color (mag)')

# Panel 3: Residual distributions
ax = axes[2]
ax.hist(obs_mag_cal - true_mag_cal, bins=30, alpha=0.5,
       color='red', label='Before', density=True)
ax.hist(corrected_mag - true_mag_cal, bins=30, alpha=0.5,
       color='blue', label='After', density=True)

# Add Gaussian fits
from scipy.stats import norm
x = np.linspace(-0.1, 0.1, 100)

# Before correction
mu1, std1 = norm.fit(obs_mag_cal - true_mag_cal)
ax.plot(x, norm.pdf(x, mu1, std1), 'r-', lw=2, alpha=0.7)

# After correction
mu2, std2 = norm.fit(corrected_mag - true_mag_cal)
ax.plot(x, norm.pdf(x, mu2, std2), 'b-', lw=2, alpha=0.7)

ax.set_xlabel('Residual (mag)')
ax.set_ylabel('Density')
ax.set_title('Residual Distributions')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Systematic Error Correction Results', fontsize=14, fontweight='bold')
save_figure(fig, 'systematic_correction')
plt.show()

print("\n✓ Demonstrated systematic error correction")

## Summary and Key Takeaways

This tutorial has demonstrated comprehensive photometric calibration techniques:

### Key Concepts

1. **Photometric Offsets**
   - Typically range from -0.02 to +0.02 mag
   - Arise from filter profiles, calibration, models
   - Critical for accurate stellar parameters

2. **Empirical Calibration**
   - Star clusters provide ideal calibrators
   - Isochrone fitting reveals systematic offsets
   - Iterative refinement improves accuracy

3. **Cross-Survey Calibration**
   - Linear transformations with color terms
   - Essential for combining multi-survey data
   - Residuals reveal systematic patterns

4. **Systematic Error Correction**
   - Complex patterns in magnitude/color space
   - Polynomial or spline fitting for corrections
   - Can achieve >2× improvement in RMS

### Best Practices

- **Always validate** calibrations with independent data
- **Check for color terms** in transformations
- **Model systematic patterns** before correcting
- **Use robust statistics** (median, MAD) for outlier rejection
- **Document all calibration** steps and coefficients

### Applications

- Harmonizing multi-survey photometry
- Improving stellar parameter accuracy
- Quality control and validation
- Creating homogeneous catalogs
- Enabling precision astrophysics

### Next Steps

- Apply these techniques to real survey data
- Combine with brutus fitting for improved results
- Develop survey-specific calibration pipelines
- Validate with spectroscopic parameters

In [ ]:
print("Tutorial 8 Complete!")
print("="*60)
print("\nGenerated plots:")
for plot_file in sorted(plots_dir.glob('*.png')):
    print(f"  - {plot_file.name}")

print("\nKey calibration results demonstrated:")
print(f"  Offset recovery:      RMS = {rms_error:.4f} mag")
print(f"  Systematic correction: {rms_before/rms_after:.2f}× improvement")
print("\nAll brutus tutorials complete!")